In [0]:
spark.sql("USE CATALOG e_comm")
spark.sql("USE SCHEMA silver")

In [0]:
spark.sql("create table if not exists e_comm.silver.order_items(order_id string, order_item_id int, product_id string, seller_id string, shipping_limit_date timestamp, price float, freight_value float) ")

In [0]:
from pyspark.sql.functions import *
import pandas as pd
import requests
from io import StringIO
from delta.tables import DeltaTable


df_source = spark.read.table("e_comm.bronze.order_items")
df_target = DeltaTable.forName(spark, "e_comm.silver.order_items")
df_target.alias("t").merge(df_source.alias("s"), "t.order_id = s.order_id and t.order_item_id = s.order_item_id")\
    .whenMatchedUpdate(
        condition = "t.product_id <> s.product_id or t.seller_id <> s.seller_id or t.shipping_limit_date <> s.shipping_limit_date or t.price <> s.price or t.freight_value <> s.freight_value",
        set = {
            "t.product_id" : "s.product_id",
            "t.seller_id" : "s.seller_id",
            "t.shipping_limit_date" : "s.shipping_limit_date",
            "t.price" : "s.price",
            "t.freight_value" : "s.freight_value"
        }
    )\
        .whenNotMatchedInsert(
            values = {
            "t.order_id" : "s.order_id",
            "t.order_item_id" : "s.order_item_id",
            "t.product_id" : "s.product_id",
            "t.seller_id" : "s.seller_id",
            "t.shipping_limit_date" : "s.shipping_limit_date",
            "t.price" : "s.price",
            "t.freight_value" : "s.freight_value"
            }
        ).execute()


In [0]:
display(spark.sql("select count(*) from e_comm.silver.order_items"))